# Notebook 3: Bayesian Logistic & Probit Regression

We build two Bayesian models using PyMC:
1. **Bayesian Logistic Regression** — logit link, Normal(0, 2.5) priors
2. **Bayesian Probit Regression** — probit link (Φ), Normal(0, 2.5) priors

Both are fitted via NUTS (No-U-Turn Sampler) and compared.

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import sys; sys.path.append('..')
from src.preprocess import full_pipeline
from src.model import build_bayesian_logistic, build_bayesian_probit, sample_model

# Leakage-safe preprocessing (split first; fit MICE/scaler on train only)
X_train, X_test, y_train, y_test, scaler, imputer, feature_names = full_pipeline('../data/framingham.csv')
print('Data loaded. Train shape:', X_train.shape)

In [ ]:
# --- Bayesian Logistic Regression (baseline prior) ---
# Baseline weakly-informative prior: beta ~ Normal(0, 2.5)
logistic_model = build_bayesian_logistic(
    X_train, y_train,
    prior_family='normal',
    beta_scale=2.5,
    model_name='bayes_logistic_baseline',
)
idata_logistic = sample_model(logistic_model, draws=3000, tune=2000, chains=4, target_accept=0.95)

# Sanity check: log-likelihood exists (needed for WAIC/LOO)
print('Has log_likelihood:', hasattr(idata_logistic, 'log_likelihood') and idata_logistic.log_likelihood is not None)

In [ ]:
# MCMC diagnostics
az.plot_trace(idata_logistic, var_names=['alpha', 'beta'])
plt.tight_layout()

In [ ]:
# Summary statistics
summary_base = az.summary(idata_logistic, var_names=['alpha', 'beta'], round_to=3)
display(summary_base)

# Prior sensitivity (shrinkage): beta ~ Normal(0, 0.5)
# Useful when VIF suggests multicollinearity (e.g., sysBP vs diaBP).
logistic_model_shrink = build_bayesian_logistic(
    X_train, y_train,
    prior_family='normal',
    beta_scale=0.5,
    model_name='bayes_logistic_shrink',
)
idata_logistic_shrink = sample_model(logistic_model_shrink, draws=2000, tune=1500, chains=4, target_accept=0.95)

summary_shrink = az.summary(idata_logistic_shrink, var_names=['alpha', 'beta'], round_to=3)
display(summary_shrink)

# Optional: compare ESS for potentially collinear blood-pressure features
if 'sysBP' in feature_names and 'diaBP' in feature_names:
    idx_sys = feature_names.index('sysBP')
    idx_dia = feature_names.index('diaBP')
    print('ESS baseline  (sysBP, diaBP):',
          float(summary_base.loc[f'beta[{idx_sys}]', 'ess_bulk']),
          float(summary_base.loc[f'beta[{idx_dia}]', 'ess_bulk']))
    print('ESS shrinkage (sysBP, diaBP):',
          float(summary_shrink.loc[f'beta[{idx_sys}]', 'ess_bulk']),
          float(summary_shrink.loc[f'beta[{idx_dia}]', 'ess_bulk']))

In [ ]:
# --- Bayesian Probit Regression ---
probit_model = build_bayesian_probit(X_train, y_train)
idata_probit = sample_model(probit_model, draws=3000, tune=2000, chains=4, target_accept=0.95)

print('Has log_likelihood:', hasattr(idata_probit, 'log_likelihood') and idata_probit.log_likelihood is not None)

In [ ]:
az.plot_trace(idata_probit, var_names=['alpha', 'beta'])
plt.tight_layout()

In [ ]:
# Save trace objects for evaluation notebook
idata_logistic.to_netcdf('../data/idata_logistic.nc')
idata_logistic_shrink.to_netcdf('../data/idata_logistic_shrink.nc')
idata_probit.to_netcdf('../data/idata_probit.nc')
print('Saved.')